<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/21_temporal_misr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Temporal MISR — one target date, nine other dates

| input | what it is |
|---|---|
| target | the LR image of the date to reconstruct |
| aux | nine Sentinel-2 RGB images of the same place on other dates (cloud <= 1 %) |
| output | HR x3 of the target date |

Each aux image is histogram-matched band by band to the target images of its scene; nothing
else is done to it.

The network is SwinIR-like.  Every attention is **target -> aux**: a target window asks, the
same window in all nine aux images answers.  There is no attention among target tokens or
among aux tokens.

The patches and the test area are the ones of the EDSR page: five validation patches with
an IKONOS HR, and Incheon, a real capture with no HR.

## 1. Setup

In [ ]:
!pip install -q timm einops
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('sr_utils.py', 'sr_models.py', 'hat_arch.py', 'hat_models.py', 'tsa_models.py', 'iqa.py'):
    urllib.request.urlretrieve(f'{BASE}/lib/{f}', f)
    sys.modules.pop(f[:-3], None)

import cv2, numpy as np, torch
import matplotlib.pyplot as plt
from sr_utils import VIS_CENTER, VIS_SIZE
from sr_models import load_edsr
from hat_models import load_hat, hat_upscale
from tsa_models import load_tswinir, run_tiled
from iqa import psnr, ssim

get = lambda p: urllib.request.urlretrieve(f'{BASE}/{p}', p.split('/')[-1])[0]
V = np.load(get('dataset/tsa/val.npz'))
T = np.load(get('dataset/tsa/test.npz'))
net = load_tswinir(get('models/21_tsa_misr/checkpoints/tswinir_x3.pt'))
edsr = load_edsr(get('models/03_edsr_x3/checkpoints/edsr_x3_v4.pt'))
hat = load_hat(get('models/07_hat_x3/checkpoints/hat_x3_v4.pth'))
NAMES = list(V['names'])


def up(a, mode=cv2.INTER_CUBIC):
    return cv2.resize(a, (a.shape[1] * 3, a.shape[0] * 3), interpolation=mode)


def run_edsr(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(edsr.parameters()).device)
    with torch.no_grad():
        return edsr(t).clamp(0, 255).round()[0].cpu().numpy().transpose(1, 2, 0).astype('uint8')


def window(i, n):
    """The zoom window of patch i — the same place the EDSR page shows."""
    cx, cy = VIS_CENTER[i]
    return (max(0, min(cy - VIS_SIZE // 2, n - VIS_SIZE)),
            max(0, min(cx - VIS_SIZE // 2, n - VIS_SIZE)), VIS_SIZE)

## 2. The inputs

One validation patch: the target, the nine aux dates, and the HR of the target date.

In [ ]:
def show_inputs(name, dates):
    g, a, h = V[f'{name}/glr'], V[f'{name}/aux'], V[f'{name}/hr']
    fig, ax = plt.subplots(1, 11, figsize=(24, 2.9))
    ax[0].imshow(g, interpolation='nearest'); ax[0].set_title('target')
    for j in range(9):
        ax[j + 1].imshow(a[j], interpolation='nearest'); ax[j + 1].set_title(dates[j], fontsize=10)
    ax[10].imshow(h); ax[10].set_title('HR')
    for x in ax:
        x.set_xticks([]); x.set_yticks([])
    fig.suptitle(name, y=1.04); plt.show()


show_inputs(NAMES[1], V['dates'][1])

## 3. Results

The five validation patches.  PSNR and SSIM against the HR are in the titles.

In [ ]:
rows = []
for i, name in enumerate(NAMES):
    g, a, h = V[f'{name}/glr'], V[f'{name}/aux'], V[f'{name}/hr']
    bic, sr = up(g), run_tiled(net, g, a)
    rows.append((name, psnr(bic, h), ssim(bic, h), psnr(sr, h), ssim(sr, h)))
    y0, x0, c = window(i, h.shape[0])
    panels = [('target', up(g, cv2.INTER_NEAREST)), ('aux (1 of 9)', up(a[0], cv2.INTER_NEAREST)),
              (f'bicubic\n{rows[-1][1]:.2f} dB  SSIM {rows[-1][2]:.3f}', bic),
              (f'TSwinIR\n{rows[-1][3]:.2f} dB  SSIM {rows[-1][4]:.3f}', sr), ('HR', h)]
    fig, ax = plt.subplots(1, 5, figsize=(20, 4.6))
    for x, (t, im) in zip(ax, panels):
        x.imshow(im[y0:y0 + c, x0:x0 + c], interpolation='nearest'); x.set_title(t)
        x.set_xticks([]); x.set_yticks([])
    fig.suptitle(name, y=1.02); plt.show()

r = np.array([x[1:] for x in rows])
print(f'{"":28s} {"bicubic":>15s} {"TSwinIR":>15s}')
for x in rows:
    print(f'{x[0]:28s} {x[1]:6.2f}  {x[2]:.3f}   {x[3]:6.2f}  {x[4]:.3f}')
print(f'{"mean":28s} {r[:, 0].mean():6.2f}  {r[:, 1].mean():.3f}   {r[:, 2].mean():6.2f}  {r[:, 3].mean():.3f}')

TSwinIR is above bicubic on all five patches, by 0.9 - 1.7 dB.  Roofs, roads and field
edges come back where bicubic leaves a blur.

The smallest gain is Athens: its first aux date is hazy and a few pixels are saturated.

## 4. Number of aux images

The same five patches with k = 1 - 9 aux images.  For each k, five random sets of k dates
are drawn and the scores averaged.

In [ ]:
rng = np.random.default_rng(0)
K = range(1, 10)
m = np.zeros((len(K), 2))
for j, k in enumerate(K):
    sc = []
    for name in NAMES:
        g, a, h = V[f'{name}/glr'], V[f'{name}/aux'], V[f'{name}/hr']
        for _ in range(5):
            sr = run_tiled(net, g, a[np.sort(rng.choice(9, k, replace=False))])
            sc.append((psnr(sr, h), ssim(sr, h)))
    m[j] = np.mean(sc, 0)
b = r[:, :2].mean(0)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for j, t in enumerate(('PSNR (dB)', 'SSIM')):
    ax[j].plot(K, m[:, j], 'o-', color='#2b6cb0', lw=2, label='TSwinIR')
    ax[j].axhline(b[j], color='#888888', lw=1.5, ls=':', label='bicubic')
    ax[j].set_xlabel('number of aux images'); ax[j].set_title(t, loc='left')
    ax[j].set_xticks(list(K)); ax[j].grid(alpha=.3)
ax[0].legend(frameon=False)
plt.tight_layout(); plt.show()

for k, (p, q) in zip(K, m):
    print(f'k={k}  {p:6.2f}  {q:.4f}')

PSNR rises from 21.19 dB with one aux image to 21.58 dB with nine, SSIM from 0.627 to 0.652.
Most of it comes with the first three (+0.26 dB); the next six add 0.13 dB.

## 5. Against EDSR and HAT

EDSR and HAT from their pages, given the same target image alone.

In [ ]:
rows2 = []
for i, name in enumerate(NAMES):
    g, a, h = V[f'{name}/glr'], V[f'{name}/aux'], V[f'{name}/hr']
    out = dict(bicubic=up(g), EDSR=run_edsr(g), HAT=hat_upscale(hat, g), TSwinIR=run_tiled(net, g, a))
    rows2.append([name] + [f(o, h) for o in out.values() for f in (psnr, ssim)])
    y0, x0, c = window(i, h.shape[0])
    fig, ax = plt.subplots(1, 5, figsize=(20, 4.6))
    for x, (t, im) in zip(ax, list(out.items()) + [('HR', h)]):
        x.imshow(im[y0:y0 + c, x0:x0 + c], interpolation='nearest')
        x.set_title(t if t == 'HR' else f'{t}\n{psnr(im, h):.2f} dB  SSIM {ssim(im, h):.3f}')
        x.set_xticks([]); x.set_yticks([])
    fig.suptitle(name, y=1.02); plt.show()

M = list(out)
r2 = np.array([x[1:] for x in rows2])
print(f'{"":28s}' + ''.join(f'{t:>16s}' for t in M))
for x in rows2:
    print(f'{x[0]:28s}' + ''.join(f'   {x[1 + 2 * j]:6.2f} {x[2 + 2 * j]:.3f}' for j in range(len(M))))
print(f'{"mean":28s}' + ''.join(f'   {r2[:, 2 * j].mean():6.2f} {r2[:, 2 * j + 1].mean():.3f}' for j in range(len(M))))

TSwinIR has the highest mean PSNR — 21.58 dB against 21.42 (EDSR) and 21.39 (HAT) — and
leads on four of the five patches; on Barcelona EDSR is 0.07 dB higher.  Mean SSIM is level
with the other two (0.652 against 0.650 and 0.655); on Athens it is the lowest of the three.

## 6. Final test — Incheon

The target is a real Sentinel-2 capture (2024-05-16); the nine aux dates are matched to it
the same way.  There is no HR, so there is no score.

In [ ]:
print('aux dates:', ', '.join(T['dates']))
for name in T['names']:
    g, a = T[f'{name}/target'], T[f'{name}/aux']
    sr = run_tiled(net, g, a)
    c = 240
    y0 = x0 = sr.shape[0] // 2 - c // 2
    panels = [('target  2024-05-16', up(g, cv2.INTER_NEAREST)), ('bicubic', up(g)),
              ('EDSR', run_edsr(g)), ('HAT', hat_upscale(hat, g)), ('TSwinIR', sr)]
    fig, ax = plt.subplots(1, 5, figsize=(20, 4.6))
    for x, (t, im) in zip(ax, panels):
        x.imshow(im[y0:y0 + c, x0:x0 + c], interpolation='nearest'); x.set_title(t)
        x.set_xticks([]); x.set_yticks([])
    fig.suptitle(name, y=1.02); plt.show()